# Modèle Sentence Transformers

Ce modèle permet d'encoder des phrases complètent en gardant leur structure grammaticale, les nuances et le contexte. Il a besoin de phrases en langage naturel pour comprendre le sens global.

### Lecture du dataset nettoyé 

In [30]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [31]:
df = pd.read_csv("../../datasets/clean/perfumes_clean.csv")
df.head()

,id,name,house,url,family,secondary_family,character,top_notes,heart_notes,base_notes,...,times_of_day,season_spring,season_summer,season_autumn,season_winter,time_day,time_evening,time_night,description,text_full
0,ysl-libre,Libre,Yves Saint Laurent,https://www.scento.com/perfumes/libre-yves-sai...,White Bloom,NaN,floral; creamy; fresh; citrusy,mandarin; neroli; lavender,orange blossom; jasmine sambac,musk; vanilla,...,day; evening,1,1,1,0,1,1,0,Mandarin and neroli brighten a creamy bloom of...,Mandarin and neroli brighten a creamy bloom of...
1,guerlain-shalimar,Shalimar,Guerlain,https://www.scento.com/perfumes/shalimar-eau-d...,Petal Garden,Velvet Sweet,floral; sweet; warm; powdery,floral notes; bergamot,iris; jasmine; rose,vanilla; opoponax; tonka bean,...,day; evening,1,1,1,0,1,1,0,Floral notes and a flash of bergamot sit above...,Floral notes and a flash of bergamot sit above...
2,tom-ford-oud-wood,Oud Wood,Tom Ford,https://www.scento.com/perfumes/oud-wood-tom-ford,Spice Route,NaN,warm; spicy; sweet; earthy,NaN,NaN,NaN,...,evening; night,0,0,1,1,0,1,1,Smooth woods and spice: cardamom and pink pepp...,Smooth woods and spice: cardamom and pink pepp...
3,creed-aventus,Aventus,Creed,https://www.scento.com/perfumes/aventus-creed,Orchard Fruit,Wood & Root,fruity; earthy; citrusy; floral,lemon; pink pepper; apple; bergamot; blackcurrant,pineapple; jasmine; patchouli,birch; ambroxan; cedarwood; oakmoss; musk,...,day; evening,1,1,1,0,1,1,0,Sparkling lemon and blackcurrant open onto pin...,Sparkling lemon and blackcurrant open onto pin...
4,mfk-baccarat-rouge-540,Baccarat Rouge 540,Maison Francis Kurkdjian,https://www.scento.com/perfumes/baccarat-rouge...,Wood & Root,White Bloom,woody; warm; floral; spicy,jasmine; saffron,cedarwood,ambergris,...,day; evening,1,0,1,1,1,1,0,"Jasmine and saffron open airy and golden, held...","Jasmine and saffron open airy and golden, held..."


### Construction des textes par thématiques pour chaque parfum

Comme vu dans la consigne, nous structurons chaque parfum en bloc thématique afin de calculer un score de couverture pondéré et explicable. Cela évite de n'utiliser que la colonne brute 'text_full'

In [32]:
# 1. Ambiance sensorielle et caractère émotionnel
bloc_ambiance = (df["description"].fillna("") + " Character: " + df["character"].fillna("")).tolist()

# 2. Famille olfactive et notes
bloc_notes = ("Olfactory family: "+df["family"].fillna("") + " " + df["secondary_family"].fillna("") + ". Notes: " + df["notes"].fillna("")).tolist()

# 3. Contexte d'utilisation (saisons, moments de la journée)
bloc_contexte = ("Season: " + df["seasons"].fillna("") + ". Time of day: " + df["times_of_day"].fillna("")+". Concentration: " + df["concentration"].fillna("")).tolist()

print("Exemple de blocs sémantiques :")
print("Bloc 1 - Ambiance sensorielle et caractère émotionnel :",bloc_ambiance[0])
print("Bloc 2 - Famille olfactive et notes :",bloc_notes[0])
print("Bloc 3 - Contexte d'utilisation (saisons, moments de la journée) :",bloc_contexte[0])

Exemple de blocs sémantiques :
Bloc 1 - Ambiance sensorielle et caractère émotionnel : Mandarin and neroli brighten a creamy bloom of orange blossom and jasmine sambac, settling into a quiet base of musk and vanilla. Character: floral; creamy; fresh; citrusy
Bloc 2 - Famille olfactive et notes : Olfactory family: White Bloom . Notes: mandarin; neroli; lavender; orange blossom; jasmine sambac; musk; vanilla
Bloc 3 - Contexte d'utilisation (saisons, moments de la journée) : Season: spring; summer; autumn. Time of day: day; evening. Concentration: eau-de-parfum


### Encodage vectoriel avec SBERT

Nous utilisons le modèle pré-entraîné 'all-MiniLM-L6-v2'. Il est optimisé pour les similarité sémentique en Anglais. C'est pour cela que notre questionnaire se fera en anglais pour simplifier la manipulation des données

In [33]:
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2893.79it/s]


### Génération et normalisation des embeddings

In [34]:
print("=== Encodage des blocs sémantiques ===")

emb_ambiance = model.encode(bloc_ambiance, convert_to_numpy=True)
emb_notes = model.encode(bloc_notes, convert_to_numpy=True)
emb_contexte = model.encode(bloc_contexte, convert_to_numpy=True)


print("Dimensions de la matrice Ambiance :", emb_ambiance.shape)
print("Dimensions de la matrice Notes :", emb_notes.shape)
print("Dimensions de la matrice Contexte :", emb_contexte.shape)

=== Encodage des blocs sémantiques ===
Dimensions de la matrice Ambiance : (1479, 384)
Dimensions de la matrice Notes : (1479, 384)
Dimensions de la matrice Contexte : (1479, 384)


### Similarité cosinus entre parfums

Nous décidons de combiner les 3 blocs avec une pondération de 45% pour l'ambiance, 35% pour les notes et 20% sur le contexte.

In [35]:

from sklearn.metrics.pairwise import cosine_similarity

#Calcul des matrices de similarité cosinus par bloc sémantique
sim_ambiance = cosine_similarity(emb_ambiance)
sim_notes = cosine_similarity(emb_notes)
sim_contexte = cosine_similarity(emb_contexte)

W_AMBIANCE = 0.45
W_NOTES = 0.35
W_CTX = 0.20
similarites_globales = (W_AMBIANCE * sim_ambiance + W_NOTES * sim_notes + W_CTX * sim_contexte)


In [36]:
def recommander_parfum(indice, n=5):
    scores = similarites_globales[indice]
    #On trie les indices des scores par ordre décroissant
    indices = np.argsort(scores)[::-1][1:n+1]  # On exclut l'indice lui-même

    resultats = df.iloc[indices][["name","house","family","character"]].copy()
    resultats["score_global"]=np.round(scores[indices], 4)
    resultats["score_ambiance"]=np.round(sim_ambiance[indice,indices], 4)
    resultats["score_notes"]=np.round(sim_notes[indice,indices], 4)
    resultats["score_contexte"]=np.round(sim_contexte[indice,indices], 4)
    return resultats

print("Parfum de référence : ", df.iloc[0]["name"], "-", df.iloc[0]["house"])
recommander_parfum(0, n=5)

Parfum de référence :  Libre - Yves Saint Laurent


,name,house,family,character,score_global,score_ambiance,score_notes,score_contexte
421,Sundazed,Byredo,White Bloom,floral; citrusy; creamy; fresh,0.8965,0.8340,0.9178,1.0000
487,Infusion de Fleur d'Oranger,Prada,White Bloom,floral; creamy; woody,0.8892,0.8219,0.9182,0.9900
1407,Neroli Portofino Parfum,Tom Ford,White Bloom,floral; creamy; citrusy; fresh,0.8707,0.8002,0.9315,0.9232
752,Connoisseur Woman,Armaf,White Bloom,floral; creamy; warm; citrusy,0.8673,0.7386,0.9568,1.0000
337,Mon Guerlain,Guerlain,White Bloom,floral; fresh; citrusy; creamy,0.8664,0.7619,0.9244,1.0000


### Test : Simulation d'une requête utilisateur 

In [37]:
#Simulation d'une requête utilisateur en anglais
requete_ambiance = ("I want a warm, sensual, sweet and enveloping fragrance with a floral"
    " touch.")

requete_notes = ("I love creamy vanilla, white flowers like jasmine, and soft musk.")

requete_contexte = "Perfect for an evening out or a romantic dinner."

notes_detestees = ["patchouli"]
moment_souhaite = "time_evening"
intensite_souhaitee = 4

#Encodage des phrases utilisateur avec SBERT
vec_user_ambiance = model.encode(requete_ambiance, convert_to_numpy=True).reshape(1, -1)
vec_user_notes = model.encode(requete_notes, convert_to_numpy=True).reshape(1, -1)
vec_user_contexte = model.encode(requete_contexte, convert_to_numpy=True).reshape(1, -1)

#Calcul des scores sémantiques par bloc
score_ambiance = cosine_similarity(vec_user_ambiance, emb_ambiance)[0]
score_notes = cosine_similarity(vec_user_notes, emb_notes)[0]
score_contexte = cosine_similarity(vec_user_contexte, emb_contexte)[0]

#Score sémantique global pondéré
score_semantique = (W_AMBIANCE * score_ambiance + W_NOTES * score_notes + W_CTX * score_contexte)

#Prise en compte des contraintes supplémentaires (notes détestées, moment souhaité, intensité souhaitée)
parfume_intensite = df["intensity"].fillna(3.0).values
score_intensite = 1 - np.abs(parfume_intensite - intensite_souhaitee) / 4.0
score_final = 0.85 * score_semantique + 0.15 * score_intensite

masque_moment = df[moment_souhaite] == 1
masque_exclusion = (df[["top_notes","heart_notes","base_notes","notes"]].apply(lambda col: col.str.contains('|'.join(notes_detestees), case=False, na=False)).any(axis=1))  # On exclut les parfums contenant les notes détestées
masque_valide = masque_moment & ~masque_exclusion


# Extraction et affichage des 5 meilleurs parfums recommandés
resultats_user = df.loc[masque_valide, ["name","house","family","notes","intensity"]].copy()
resultats_user["score_final"] = np.round(score_final[masque_valide], 4)
resultats_user["bloc_ambiance"] = np.round(score_ambiance[masque_valide], 4)
resultats_user["bloc_notes"] = np.round(score_notes[masque_valide], 4)
resultats_user["bloc_contexte"] = np.round(score_contexte[masque_valide], 4)

top5 = resultats_user.sort_values(by="score_final", ascending=False).head(5)
print("=== Top 5 des parfums recommandés ===")
print(top5)

=== Top 5 des parfums recommandés ===
                         name               house        family  \
562                      Haya             Lattafa   White Bloom   
1463    Black Opium Le Parfum  Yves Saint Laurent  Velvet Sweet   
1281         Vanilla Extasy :             Montale   White Bloom   
1067      Devotion Intense, .       Dolce&Gabbana  Velvet Sweet   
823   Odyssey Eau de Montagne               Armaf  Velvet Sweet   

                                                  notes  intensity  \
562   champagne; strawberry; mandarin; blood orange;...        NaN   
1463                      floral notes; vanilla; coffee        4.0   
1281  bergamot; peach; apricot; lavender; jasmine; y...        3.0   
1067                  hazelnut; orange blossom; vanilla        3.0   
823   almond; lactones; spice; white flowers; vanill...        NaN   

      score_final  bloc_ambiance  bloc_notes  bloc_contexte  
562        0.5734         0.6621      0.6116         0.1508  
1463       0.5